# Task 01: Automated Historical Data Preparation Orchestrator
### CYCLO-NEXUS: Autonomous Tropical Cyclone Intelligence Platform
**Owner:** Agent ALPHA (Tasks 1–5: Ingestion, Reprojection, Calibration, Best Track Harmonization)

This notebook automates downloading and preprocessing all required historical training data:
1. **NOAA IBTrACS & IMD Best Track:** Automatically pulls North Indian Ocean best tracks, parses RSMC New Delhi intensity/coordinates, and exports clean CSVs & Parquets.
2. **Multi-Spectral Satellite Imagery:** Automatically fetches and fuses ISRO MOSDAC (INSAT-3D/3DR) and JAXA GSMaP into unified `(4, 1024, 1024)` float32 tensors.
3. **Dataset Verification:** Validates tensor bounds `[0.0, 1.0]` and chronological sequence continuity before feeding Model 1 (`02_yolo_obb_training.ipynb`) and Model 2 (`03_spatiotemporal_training.ipynb`).

In [ ]:
# Step 1: Install required geospatial and data processing dependencies
!pip install requests pandas numpy h5py scipy pyarrow tqdm

In [ ]:
from google.colab import drive
import sys
from pathlib import Path

# Step 2: Mount Google Drive
drive.mount('/content/drive')

# Point to project root
PROJECT_ROOT = Path('/content/drive/MyDrive/cyclonegaurd')
sys.path.append(str(PROJECT_ROOT))

print(f"Mounted Google Drive. Project root: {PROJECT_ROOT}")

## 1. Automated Historical Best Track Ingestion (IBTrACS + IMD)
Downloads the official NOAA IBTrACS North Indian Ocean archive, filters for benchmark storms (Amphan, Tauktae, Biparjoy, Fani, Michaung), and exports standardized CSVs matching the `IMDTrackHarmonizer` specification.

In [ ]:
# Set correct NOAA URL and initialize track downloader
import data_pipeline.catalog.historical_track_downloader as htd
CORRECT_URL = 'https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r00/access/csv/ibtracs.NI.list.v04r00.csv'
htd.IBTRACS_NI_URL = CORRECT_URL
if hasattr(htd, 'IBTRACS_FALLBACK_URLS'):
    htd.IBTRACS_FALLBACK_URLS[0] = CORRECT_URL

besttrack_dir = PROJECT_ROOT / 'data' / 'besttrack'
besttrack_dir.mkdir(parents=True, exist_ok=True)
track_downloader = htd.HistoricalTrackDownloader(cache_dir=besttrack_dir)
track_downloader.raw_csv_path = besttrack_dir / 'ibtracs.NI.list.v04r00.csv'

print("Starting automated IBTrACS Best Track download & extraction...")
extracted_tracks = track_downloader.download_and_extract_benchmark_cyclones()

print("\n--- Extracted Historical Cyclone Tracks ---")
for storm_name, file_path in extracted_tracks.items():
    print(f"  • {storm_name}: {file_path}")

## 2. Automated Multi-Spectral Satellite Batch Ingestion
Chronologically steps through historical cyclone windows in 30-minute intervals, fetching:
- **Channel 0:** INSAT-3D/3DR TIR-1 (10.8 µm)
- **Channel 1:** INSAT-3D/3DR TIR-2 (12.0 µm)
- **Channel 2:** INSAT-3D/3DR WV (6.9 µm)
- **Channel 3:** JAXA GSMaP Precipitation

Fuses and reprojects to `(4, 1024, 1024)` normalized arrays `[0.0, 1.0]`.

In [ ]:
from data_pipeline.ingestion.historical_satellite_downloader import HistoricalSatelliteDownloader

tensors_dir = PROJECT_ROOT / 'data' / 'tensors'
raw_dir = PROJECT_ROOT / 'data' / 'raw'

sat_downloader = HistoricalSatelliteDownloader(
    output_dir=tensors_dir,
    raw_dir=raw_dir
)

# Download and assemble tensors for benchmark storms
target_storms = ["AMPHAN", "TAUKTAE", "BIPARJOY"]

for storm in target_storms:
    print(f"\nProcessing satellite timeline for Cyclone {storm}...")
    assembled = sat_downloader.download_benchmark_storm(storm, interval_minutes=30, max_frames=24)
    print(f"  Successfully generated {len(assembled)} tensors for {storm}.")

## 3. Dataset Integrity & Readiness Verification
Verifies that all tensors strictly meet the AI model requirements before starting training.

In [ ]:
import numpy as np
from pathlib import Path
from forecaster.data.temporal_matrix_builder import TemporalMatrixBuilder

tensor_files = sorted(list(tensors_dir.glob('*.npy')))
print(f"Total assembled tensors available: {len(tensor_files)}")

if len(tensor_files) > 0:
    sample_tensor = np.load(tensor_files[0])
    print(f"Sample Tensor: {tensor_files[0].name}")
    print(f"  • Shape: {sample_tensor.shape} (Expected: (4, 1024, 1024))")
    print(f"  • Dtype: {sample_tensor.dtype} (Expected: float32)")
    print(f"  • Value Range: [{sample_tensor.min():.3f}, {sample_tensor.max():.3f}] (Expected: [0.0, 1.0])")
    
    assert sample_tensor.shape == (4, 1024, 1024), "Tensor shape mismatch!"
    assert 0.0 <= sample_tensor.min() and sample_tensor.max() <= 1.0, "Normalization error!"
    
    # Verify temporal sequence formation for ConvLSTM
    sequences = TemporalMatrixBuilder.build_sequences(tensor_files, seq_length=6)
    print(f"  • Valid 6-step temporal sequences generated: {len(sequences)}")
    print("\n Dataset is verified and ready for 02_yolo_obb_training.ipynb and 03_spatiotemporal_training.ipynb!")
else:
    print("No tensors found in directory.")